In [ ]:
import zipfile
import io
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

In [22]:
zip_path = '/content/drive/My Drive/evaluated_positions.zip'
extract_path = '/content/extracted_positions'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

In [23]:
!pip install chess

In [24]:
"""
Convert FEN strings to tensor representations for neural network input.
"""

import chess
import numpy as np
import torch

import json

class ChessDataset(torch.utils.data.Dataset):
    def __init__(self, jsonl_path, use_attack_map=False):
        self.use_attack_map = use_attack_map
        self.data_entries = []

        with open(jsonl_path, 'r') as f:
            for line in f:
                line = line.strip()
                if line:
                    self.data_entries.append(json.loads(line))

    def __len__(self):
        return len(self.data_entries)

    def __getitem__(self, idx):
        entry = self.data_entries[idx]

        # Generate the features
        X_tensor = fen_to_tensor(entry['fen'], self.use_attack_map)

        # Extract the target label as a float32 tensor
        y_target = torch.tensor(entry['target'], dtype=torch.float32)

        return X_tensor, y_target

def fen_to_tensor(fen, use_attack_map=False):
    board = chess.Board(fen)
    channels = 22 if use_attack_map else 20
    tensor = np.zeros((channels, 8, 8), dtype=np.float32)
    for square in chess.SQUARES:
        piece = board.piece_at(square)
        if piece is not None:
            color_idx = 0 if piece.color == chess.WHITE else 1
            channel = (piece.piece_type - 1) + (color_idx * 6)
            rank, col = divmod(square, 8)
            row = 7 - rank
            tensor[channel, row, col] = 1.0

    tensor[12, :, :] = int(board.turn)  # Turn channel
    tensor[13, :, :] = 1.0 if board.has_kingside_castling_rights(chess.WHITE) else 0.0  # White kingside castling
    tensor[14, :, :] = 1.0 if board.has_queenside_castling_rights(chess.WHITE) else 0.0  # White queenside castling
    tensor[15, :, :] = 1.0 if board.has_kingside_castling_rights(chess.BLACK) else 0.0  # Black kingside castling
    tensor[16, :, :] = 1.0 if board.has_queenside_castling_rights(chess.BLACK) else 0.0  # Black queenside castling

    if board.ep_square is not None:
        ep_row, ep_col = divmod(board.ep_square, 8)
        tensor[17, ep_row, ep_col] = 1.0  # En passant square
    tensor[18, :, :] = board.halfmove_clock / 100.0  # Halfmove clock
    tensor[19, :, :] = min(board.fullmove_number, 200) / 200.0  # Fullmove number

    if use_attack_map:
        attack_map = np.zeros((2, 8, 8), dtype=np.float32)
        for square in chess.SQUARES:
            rank, col = divmod(square, 8)
            row = 7 - rank

            attack_map[0, row, col] = 1.0 if board.attackers(chess.WHITE, square) else 0.0
            attack_map[1, row, col] = 1.0 if board.attackers(chess.BLACK, square) else 0.0
        tensor[20:22, :, :] = attack_map

    return torch.tensor(tensor)



In [25]:
import torch
import torch.nn as nn

class ResBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ResBlock, self).__init__()
        # not for this resnet we are not downsampling, so stride is always 1

        # conv layer 1
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        # conv layer 2
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)

        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1),
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.shortcut = nn.Identity()


    def forward(self, x):
        residual = x
        # pass through layer 1
        out = self.relu(self.bn1(self.conv1(x)))
        # pass through layer 2
        out = self.bn2(self.conv2(out))
        # residual connection
        out += self.shortcut(residual)
        # apply ReLU activation
        out = self.relu(out)

        return out


class ChessResNet(nn.Module):
    def __init__(self, input_channels=20, num_blocks=5, num_classes=1):
        super().__init__()
        self.input_channels = input_channels
        self.num_blocks = num_blocks
        self.num_classes = num_classes

        # Initial convolutional layer
        self.conv1 = nn.Conv2d(input_channels, 64, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)

        # Residual blocks
        self.res_blocks = self._make_layer(64, 64, num_blocks)

        # Compressing values with 1x1 convolution to reduce the number of channels before flattening
        self.value_conv = nn.Conv2d(64, 2, kernel_size=1)
        self.value_bn = nn.BatchNorm2d(2)
        self.relu_value = nn.ReLU(inplace=True)

        # Fully connected layer for output
        self.fc = nn.Sequential(
            nn.Linear(2 * 8 * 8, 64),  # Flattened size after conv layers
            nn.ReLU(inplace=True),
            nn.Linear(64, num_classes),
            nn.Sigmoid()  # Use Sigmoid for binary output (0 to 1)
        )

    def _make_layer(self, in_channels, out_channels, num_blocks):
        layers = [ResBlock(in_channels, out_channels)]
        for _ in range(1, num_blocks):
            layers.append(ResBlock(out_channels, out_channels))
        return nn.Sequential(*layers)

    def forward(self, x):
        # Initial convolutional layer
        out = self.relu(self.bn1(self.conv1(x)))

        # Pass through residual blocks
        out = self.res_blocks(out)

        # Compressing values with 1x1 convolution
        out = self.relu_value(self.value_bn(self.value_conv(out)))

        # Flatten the output for the fully connected layer
        out = out.view(out.size(0), -1)

        # Fully connected layers
        out = self.fc(out)

        return out

In [26]:
from torch.utils.data import random_split, DataLoader

In [27]:
batch_size = 1024

In [28]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ChessResNet(input_channels=20).to(device)
dataset = ChessDataset("/content/extracted_positions/evaluated_positions.jsonl", use_attack_map=False)

total_size = len(dataset)
train_size = int(0.8 * total_size)
val_size = int(0.1 * total_size)
test_size = total_size - train_size - val_size
train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=batch_size, num_workers=2, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, num_workers=2, shuffle=False, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, num_workers=2, shuffle=False, pin_memory=True)


In [29]:
print(len(train_loader), len(val_loader))

7813 977


In [30]:
# hyper params
epochs = 20
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=0.01, pct_start=0.3, steps_per_epoch=len(train_loader), epochs=epochs, anneal_strategy='cos')

In [ ]:
# Training loop
for k in range(epochs):
    running_loss = 0.0
    model.train()
    for i, (inputs, targets) in enumerate(train_loader):
        inputs, targets = inputs.to(device), targets.to(device)

        # zero the gradients
        optimizer.zero_grad()

        # forward pass
        outputs = model(inputs)

        # compute loss
        loss = criterion(outputs.view(-1), targets)

        #backpropagation
        loss.backward()

        # update weights
        optimizer.step()
        scheduler.step()

        running_loss += loss.item()
        if i % 1000 == 999:  # print every 1000 mini-batches
            print(f"[{k + 1}, {i + 1}] loss: {running_loss / 1000:.6f}")
            running_loss = 0.0
    print(f"Finished epoch {k + 1}")\
    # save weights every epoch
    torch.save(model.state_dict(), f"/content/drive/MyDrive/ChessAI/chess_resnet_epoch_{k+1}.pt")
    print(f"Epoch {k+1} complete. Weights cached to Drive.")
print("Finished Training")

[1, 1000] loss: 0.007535
[1, 2000] loss: 0.005847
[1, 3000] loss: 0.005532
[1, 4000] loss: 0.005308
[1, 5000] loss: 0.005183
[1, 6000] loss: 0.005082
[1, 7000] loss: 0.005023
Finished epoch 1
Epoch 1 complete. Weights cached to Drive.
[2, 1000] loss: 0.004878
[2, 2000] loss: 0.004814
[2, 3000] loss: 0.004809
[2, 4000] loss: 0.004681
[2, 5000] loss: 0.004700
[2, 6000] loss: 0.004686
[2, 7000] loss: 0.004585
Finished epoch 2
Epoch 2 complete. Weights cached to Drive.
[3, 1000] loss: 0.004573
[3, 2000] loss: 0.004469
[3, 3000] loss: 0.004478
[3, 4000] loss: 0.004413
[3, 5000] loss: 0.004444
[3, 6000] loss: 0.004368
[3, 7000] loss: 0.004348
Finished epoch 3
Epoch 3 complete. Weights cached to Drive.
[4, 1000] loss: 0.004250
[4, 2000] loss: 0.004324
[4, 3000] loss: 0.004256
[4, 4000] loss: 0.004256
[4, 5000] loss: 0.004266
[4, 6000] loss: 0.004216
[4, 7000] loss: 0.004233
Finished epoch 4
Epoch 4 complete. Weights cached to Drive.
[5, 1000] loss: 0.004194
[5, 2000] loss: 0.004135
[5, 3000] 

In [ ]:
import torch

# 1. Re-instantiate your device, model, and dataset elements exactly as before
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ChessResNet(input_channels=20).to(device)

# 2. LOAD YOUR CACHED WEIGHTS
checkpoint_path = "/content/drive/MyDrive/ChessAI/chess_resnet_epoch_10.pt"
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
print("💪 Successfully loaded Epoch 10 weights from Google Drive!")

# 3. Re-create your dataloaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, num_workers=2, shuffle=True, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, num_workers=2, shuffle=False, pin_memory=True)

# 4. Re-instantiate optimization heads
epochs = 20  # Keep your original total goal
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)

# Re-create the scheduler exactly as it was originally defined
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=0.015,
    steps_per_epoch=len(train_loader),
    epochs=epochs,
    pct_start=0.3,
    anneal_strategy='cos'
)

# 5. THE MAGIC STEP: Fast-forward the scheduler to the start of Epoch 11
# This calculates exactly how many optimization steps occurred in the first 10 epochs
completed_epochs = 10
total_steps_to_skip = completed_epochs * len(train_loader)

print(f"⏩ Fast-forwarding scheduler past {total_steps_to_skip:,} completed steps...")
for _ in range(total_steps_to_skip):
    # This advances the scheduler learning rate/momentum curves without updating model weights
    scheduler.step()

# --- RESUME TRAINING LOOP ---
# Start the loop index at completed_epochs (10, which prints as Epoch 11)
for k in range(completed_epochs, epochs):
    running_loss = 0.0
    model.train()
    for i, (inputs, targets) in enumerate(train_loader):
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs.view(-1), targets)
        loss.backward()

        optimizer.step()
        scheduler.step()  # Tracks smoothly along the remaining half of the cosine curve

        running_loss += loss.item()
        if i % 250 == 249:
            print(f"[{k + 1}, {i + 1}] loss: {running_loss / 250:.6f}")
            running_loss = 0.0

    print(f"Finished epoch {k + 1}")

    # Save the new checkpoints back to Drive
    torch.save(model.state_dict(), f"/content/drive/MyDrive/ChessAI/chess_resnet_epoch_{k + 1}.pt")
    print(f"Epoch {k + 1} complete. Weights cached to Drive.")

print("🎉 Finished Training Remaining Epochs!")

In [ ]:
model.eval()
total_val_loss = 0.0
num_batches = 0
with torch.no_grad():
    for inputs, targets in val_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        loss = criterion(outputs.view(-1), targets)
        total_val_loss += loss.item()
        num_batches += 1


print(f"Total validation loss: {total_val_loss / num_batches:.6f}")

In [31]:
# loss with use_attack_map= False: 0.00564, train: 0.0049
# loss with use_attack_map= True:0.00562, train:0.0040

In [32]:
total_params = sum(p.numel() for p in model.parameters())
print(total_params)

390727
